<h2> Aniqa's Notebook for SQL portion</h2>

Guiding Question: Which airlines have the fastest service (quickness of getting on and off the air, maybe lack of delays)?

<h5> Setup

In [1]:
import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import sqlite3

In [2]:
conn = sqlite3.connect("flights.db")

In [3]:
airlines = pd.read_csv("airlines.csv")
airports = pd.read_csv("airports.csv")
flights = pd.read_csv("flights.csv", low_memory=False)

In [4]:
airlines.to_sql("airlines", conn, if_exists="replace", index=False)
airports.to_sql("airports", conn, if_exists="replace", index=False)
flights.to_sql("flights", conn, if_exists="replace", index=False) #big dataset, takes a while to load

5819079

<h3> Running Basic Queries </h3>
To figure out schema And foreign/primary keys for joins 

In [5]:
pd.read_sql("SELECT * FROM airlines;", conn) #primary_key is IATA_CODE

,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways
5,OO,Skywest Airlines Inc.
6,AS,Alaska Airlines Inc.
7,NK,Spirit Air Lines
8,WN,Southwest Airlines Co.
9,DL,Delta Air Lines Inc.


In [6]:
pd.read_sql("SELECT * FROM airports;", conn) #primary_key is IATA_CODE

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
...,...,...,...,...,...,...,...
317,WRG,Wrangell Airport,Wrangell,AK,USA,56.48433,-132.36982
318,WYS,Westerly State Airport,West Yellowstone,MT,USA,44.68840,-111.11764
319,XNA,Northwest Arkansas Regional Airport,Fayetteville/Springdale/Rogers,AR,USA,36.28187,-94.30681
320,YAK,Yakutat Airport,Yakutat,AK,USA,59.50336,-139.66023


In [7]:
pd.read_sql("SELECT * FROM flights LIMIT 5;", conn) #primary_key is flight number, 
#foreign key is AIRLINE, which references IATA_CODE in airlines
#foreign key is ORIGIN_AIRPORT, DESTINATION_AIRPORT and it references IATA_CODE in 


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,None,None,None,None,None,None
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,None,None,None,None,None,None
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,None,None,None,None,None,None
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,None,None,None,None,None,None
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,None,None,None,None,None,None


<h3>Query to see all columns of <b>flights</b> table</h3>

In [8]:
pd.read_sql_query("""
PRAGMA table_info(flights);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,YEAR,INTEGER,0,None,0
1,1,MONTH,INTEGER,0,None,0
2,2,DAY,INTEGER,0,None,0
3,3,DAY_OF_WEEK,INTEGER,0,None,0
4,4,AIRLINE,TEXT,0,None,0
5,5,FLIGHT_NUMBER,INTEGER,0,None,0
6,6,TAIL_NUMBER,TEXT,0,None,0
7,7,ORIGIN_AIRPORT,TEXT,0,None,0
8,8,DESTINATION_AIRPORT,TEXT,0,None,0
9,9,SCHEDULED_DEPARTURE,INTEGER,0,None,0


<h2> Main Queries to Answer Guiding Question

IMPORTANT: Negative values in DEPARTURE_DELAYS and ARRIVAL_DELAYS means the flight arrived **early** and weren't actually delays. Positive values in the column means it's a delay.

<h2> Analyzing Departures

<h5> Null Handling for Departures 

**Case 0: Null Values Between Departures and Arrivals**

This query is seeing if there's instances of flights where there is a departure time but no arrival time. 

In [9]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE DEPARTURE_TIME IS NOT NULL AND ARRIVAL_TIME IS NULL;''', conn)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,MQ,2758,N939MQ,ROW,DFW,1110,...,None,None,0,1,B,None,None,None,None,None
1,2015,1,1,4,EV,4654,N29515,IAH,HRL,1125,...,None,None,1,0,None,None,None,None,None,None
2,2015,1,1,4,EV,4503,N14514,IAH,BRO,1134,...,None,None,0,1,B,None,None,None,None,None
3,2015,1,1,4,EV,4654,N14543,CRP,HRL,1410,...,None,None,1,0,None,None,None,None,None,None
4,2015,1,1,4,EV,4264,N27962,STL,IAH,1423,...,None,None,0,1,C,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6355,2015,12,31,4,EV,2749,N676AE,DFW,BRO,1155,...,None,None,1,0,None,None,None,None,None,None
6356,2015,12,31,4,UA,1291,N596UA,LIH,LAX,1405,...,None,None,1,0,None,None,None,None,None,None
6357,2015,12,31,4,EV,2785,N684JW,DFW,BRO,1530,...,None,None,1,0,None,None,None,None,None,None
6358,2015,12,31,4,WN,1754,N515SW,AUS,LBB,1605,...,None,None,0,1,A,None,None,None,None,None


This is perhaps a case of incomplete flight logs. For the purposes of this project, let's not consider flights where there is a departure time but no arrival time. 

In [10]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE DEPARTURE_TIME IS NULL AND ARRIVAL_TIME IS NOT NULL;''', conn) 

#case where arrival time exists but departure time doesn't
#thankfully, no weird cases like this exists

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY


**Case 1: BOTH Departure Times and Departure Delays are Null**

In [11]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE DEPARTURE_TIME IS NULL AND DEPARTURE_DELAY IS NULL;''', conn) 

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,136,N431AS,ANC,SEA,135,...,None,None,0,1,A,None,None,None,None,None
1,2015,1,1,4,AA,2459,N3BDAA,PHX,DFW,200,...,None,None,0,1,B,None,None,None,None,None
2,2015,1,1,4,OO,5254,N746SK,MAF,IAH,510,...,None,None,0,1,B,None,None,None,None,None
3,2015,1,1,4,MQ,2859,N660MQ,SGF,DFW,525,...,None,None,0,1,B,None,None,None,None,None
4,2015,1,1,4,OO,5460,N583SW,RDD,SFO,530,...,None,None,0,1,A,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86148,2015,12,31,4,UA,1789,None,IAH,TPA,1955,...,None,None,0,1,A,None,None,None,None,None
86149,2015,12,31,4,UA,222,None,SFO,LAX,2000,...,None,None,0,1,A,None,None,None,None,None
86150,2015,12,31,4,AA,2245,N880AA,MIA,SAN,2019,...,None,None,0,1,A,None,None,None,None,None
86151,2015,12,31,4,NK,416,N522NK,FLL,IAG,2155,...,None,None,0,1,A,None,None,None,None,None


In [12]:
pd.read_sql_query('''SELECT SUM(CANCELLED), SUM(DIVERTED)
            FROM flights f
            WHERE DEPARTURE_TIME IS NULL AND DEPARTURE_DELAY IS NULL;''', conn) 

,SUM(CANCELLED),SUM(DIVERTED)
0,86153,0


**Case 2: Departure Time Exists But Departure Delay is Null**

In [13]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE DEPARTURE_TIME IS NOT NULL AND DEPARTURE_DELAY IS NULL;''', conn) 

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY


**Case 3: Departure Delay Exists But Departure Time is Null**

In [14]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE DEPARTURE_TIME IS NULL AND DEPARTURE_DELAY IS NOT NULL;''', conn) 


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY


<h4> Departures: Joins and Aggregations to Answer Question

The WHERE clause is filtering out any flights that were either cancelled or was incomplete (meaning there was a departure time but no arrival time).

In [15]:
#this query is done to successfully carry out the join between flights and airlines tables
pd.read_sql_query('''SELECT 
            a.AIRLINE, f.DEPARTURE_TIME, f.DEPARTURE_DELAY
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL AND DEPARTURE_TIME IS NOT NULL AND ARRIVAL_TIME IS NOT NULL
            ORDER BY CAST(DEPARTURE_DELAY AS INTEGER)
            LIMIT 10;''', conn)

,AIRLINE,DEPARTURE_TIME,DEPARTURE_DELAY
0,Alaska Airlines Inc.,1553.0,-82.0
1,American Airlines Inc.,2042.0,-68.0
2,Delta Air Lines Inc.,559.0,-61.0
3,Skywest Airlines Inc.,921.0,-56.0
4,Atlantic Southeast Airlines,1310.0,-55.0
5,Delta Air Lines Inc.,1817.0,-52.0
6,Skywest Airlines Inc.,647.0,-48.0
7,Alaska Airlines Inc.,1742.0,-48.0
8,Alaska Airlines Inc.,1743.0,-47.0
9,Frontier Airlines Inc.,644.0,-46.0


In [16]:
#query is looking at average delay (in minutes) for each airline, sorted in ascending order
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL AND DEPARTURE_TIME IS NOT NULL AND ARRIVAL_TIME IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

,AIRLINE,Average_Delay
0,Hawaiian Airlines Inc.,0.475208
1,Alaska Airlines Inc.,1.730191
2,US Airways Inc.,6.103964
3,Delta Air Lines Inc.,7.345122
4,Skywest Airlines Inc.,7.749692
5,Atlantic Southeast Airlines,8.642840
6,American Airlines Inc.,8.848322
7,Virgin America,9.004970
8,American Eagle Airlines Inc.,9.995351
9,Southwest Airlines Co.,10.535050


In [17]:
#query is looking at average delay (in minutes) for each airline, sorted in ascending order
#looking at non-zero delays (meaning pure delays, no early departures)
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY > 0 AND DEPARTURE_DELAY IS NOT NULL 
                AND DEPARTURE_TIME IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

,AIRLINE,Average_Delay
0,Hawaiian Airlines Inc.,16.803406
1,Alaska Airlines Inc.,25.879313
2,Southwest Airlines Co.,26.865806
3,US Airways Inc.,28.420968
4,Delta Air Lines Inc.,29.621197
5,Virgin America,30.250642
6,United Air Lines Inc.,32.464701
7,American Airlines Inc.,34.253064
8,JetBlue Airways,37.551519
9,Skywest Airlines Inc.,39.108332


In [18]:
#query is looking at minimum delay (in minutes) for each airline, sorted in ascending order
#minimum for each airline means the earliest an airline has landed before its scheduled time
#ex: -60 minutes means the flight departed 60 minutes EARLIER than scheduled
#ascending order is appropriate here since we are dealing with negative values
pd.read_sql_query('''SELECT 
            a.AIRLINE, MIN(f.DEPARTURE_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY MIN(f.DEPARTURE_DELAY) ASC
            ;''', conn)

,AIRLINE,MIN(f.DEPARTURE_DELAY)
0,Alaska Airlines Inc.,-82.0
1,American Airlines Inc.,-68.0
2,Delta Air Lines Inc.,-61.0
3,Skywest Airlines Inc.,-56.0
4,Atlantic Southeast Airlines,-55.0
5,Frontier Airlines Inc.,-46.0
6,United Air Lines Inc.,-40.0
7,Spirit Air Lines,-37.0
8,American Eagle Airlines Inc.,-36.0
9,US Airways Inc.,-35.0


In [19]:
#query is looking at maximum delay (in minutes) for each airline, sorted in descending order
pd.read_sql_query('''SELECT 
            a.AIRLINE, MAX(f.DEPARTURE_DELAY), MAX(f.DEPARTURE_DELAY)/60 AS MAX_in_hrs
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY MAX(f.DEPARTURE_DELAY) DESC
            ;''', conn)

,AIRLINE,MAX(f.DEPARTURE_DELAY),MAX_in_hrs
0,American Airlines Inc.,1988.0,33.133333
1,American Eagle Airlines Inc.,1544.0,25.733333
2,Hawaiian Airlines Inc.,1433.0,23.883333
3,Skywest Airlines Inc.,1378.0,22.966667
4,United Air Lines Inc.,1314.0,21.900000
5,Delta Air Lines Inc.,1289.0,21.483333
6,Atlantic Southeast Airlines,1274.0,21.233333
7,Frontier Airlines Inc.,1112.0,18.533333
8,JetBlue Airways,1006.0,16.766667
9,Alaska Airlines Inc.,963.0,16.050000


<h5> Conclusion: </h5>

The following is only for flights that were not cancelled. 

3 airlines with the least average departure delay, including departures that happened earlier than scheduled, were Hawaiian, Alaskan, and US Airlines. 

3 airlines with least average delays (counting only pure delays, no early departures included) were Hawaiian, Alaskan, and Southwest Airlines. 

3 earliest departures recorded that happened before scheduled departure was Alaskan, American, and Delta airlines.

3 airlines with the longest departure delays was American, American Eagle, and Hawaiian airlines. 

After performing these aggregations, Hawaiian and Alaskan airlines are among the fastest and reliable options for departures. Reasons why this could be true is because these airlines might not be as big and expansive as other options (such as Delta or United), which could mean they have lower amounts of data and thus a better average. They could also be in locations with better weather or have better aircrafts that have less maintenance issues. However, from an elementary standpoint with these basic analysis, Hawaiian and Alaskan might be the best options. 

<h2> Analyzing Arrivals

<h5> Null Value Handling for Arrivals

**Case 1: Null Value for BOTH Arrival Time and Arrival Delay**

In [20]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE ARRIVAL_TIME IS NULL AND ARRIVAL_DELAY IS NULL
            ;''', conn)


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,136,N431AS,ANC,SEA,135,...,None,None,0,1,A,None,None,None,None,None
1,2015,1,1,4,AA,2459,N3BDAA,PHX,DFW,200,...,None,None,0,1,B,None,None,None,None,None
2,2015,1,1,4,OO,5254,N746SK,MAF,IAH,510,...,None,None,0,1,B,None,None,None,None,None
3,2015,1,1,4,MQ,2859,N660MQ,SGF,DFW,525,...,None,None,0,1,B,None,None,None,None,None
4,2015,1,1,4,OO,5460,N583SW,RDD,SFO,530,...,None,None,0,1,A,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92508,2015,12,31,4,UA,1789,None,IAH,TPA,1955,...,None,None,0,1,A,None,None,None,None,None
92509,2015,12,31,4,UA,222,None,SFO,LAX,2000,...,None,None,0,1,A,None,None,None,None,None
92510,2015,12,31,4,AA,2245,N880AA,MIA,SAN,2019,...,None,None,0,1,A,None,None,None,None,None
92511,2015,12,31,4,NK,416,N522NK,FLL,IAG,2155,...,None,None,0,1,A,None,None,None,None,None


When both arrival time and arrival delay is NULL, this leads to the natural assumption that the flight was cancelled to begin with. If a flight was cancelled, there's a 1 in the CANCELLED column. The WHERE clause for the preceding query yielded 92513 rows (meaning 92513 rows had NULL in both arrival time and arrival delay). If I sum the CANCELLATION column with this same predicate and get 92513 as the output, it means null arrival times and arrival delays are cancelled flights. I'll carry this out in the next cell. 

In [21]:
pd.read_sql_query('''SELECT SUM(CANCELLED)
            FROM flights f
            WHERE ARRIVAL_TIME IS NULL AND ARRIVAL_DELAY IS NULL
            ;''', conn)

,SUM(CANCELLED)
0,89884


Since 89884 out of 92513 were indeed cancelled, the assumption I wrote above is mostly true. However, let's analyze the rows where the flights weren't cancelled. 

In [22]:
92513 - 89884

2629

In [23]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE ARRIVAL_TIME IS NULL AND ARRIVAL_DELAY IS NULL AND CANCELLED != 1
            ;''', conn)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,EV,4654,N29515,IAH,HRL,1125,...,None,None,1,0,None,None,None,None,None,None
1,2015,1,1,4,EV,4654,N14543,CRP,HRL,1410,...,None,None,1,0,None,None,None,None,None,None
2,2015,1,1,4,OO,5488,N791SK,IAH,ASE,1427,...,None,None,1,0,None,None,None,None,None,None
3,2015,1,1,4,OO,5203,N774SK,LAX,ASE,1816,...,None,None,1,0,None,None,None,None,None,None
4,2015,1,2,5,EV,2548,N905EV,DFW,BTR,830,...,None,None,1,0,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2624,2015,12,30,3,WN,870,N757LV,LAS,AMA,1925,...,None,None,1,0,None,None,None,None,None,None
2625,2015,12,31,4,EV,2765,N671AE,DFW,BRO,855,...,None,None,1,0,None,None,None,None,None,None
2626,2015,12,31,4,EV,2749,N676AE,DFW,BRO,1155,...,None,None,1,0,None,None,None,None,None,None
2627,2015,12,31,4,UA,1291,N596UA,LIH,LAX,1405,...,None,None,1,0,None,None,None,None,None,None


This means 2629 flights had no arrival time or arrival delay but also wasn't cancelled. What other reasons could possibly lead to no arrival time or arrival delay if it wasn't cancelled? Let's look at DIVERTED column.

In [24]:
pd.read_sql_query('''SELECT SUM(DIVERTED)
            FROM flights f
            WHERE ARRIVAL_TIME IS NULL AND ARRIVAL_DELAY IS NULL AND CANCELLED != 1
            ;''', conn)

,SUM(DIVERTED)
0,2629


So that means an overwhelming majority of flights where arrival time and arrival delays didn't exist were cancelled and the remaining were diverted. 

**Case 2: Arrival Time Exists But Arrival Delay is Null**

To clarify, a NULL value in arrival delay when an arrival time exists DOES NOT MEAN there was no delay. If there was no delay, the value would be 0 so NULL values in arrival delay when there is an arrival time means something else. 

In [25]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE ARRIVAL_TIME IS NOT NULL AND ARRIVAL_DELAY = 0
            ;''', conn)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,695,N607AS,GEG,SEA,500,...,610.0,0.0,0,0,None,None,None,None,None,None
1,2015,1,1,4,OO,7404,N8982A,HIB,MSP,510,...,618.0,0.0,0,0,None,None,None,None,None,None
2,2015,1,1,4,OO,6512,N925SW,FAT,LAX,535,...,650.0,0.0,0,0,None,None,None,None,None,None
3,2015,1,1,4,DL,2453,N966DL,GSP,ATL,550,...,649.0,0.0,0,0,None,None,None,None,None,None
4,2015,1,1,4,EV,5978,N21129,FAR,ORD,600,...,803.0,0.0,0,0,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126208,2015,12,31,4,OO,5600,N776SK,SFO,MFR,2246,...,7.0,0.0,0,0,None,None,None,None,None,None
126209,2015,12,31,4,F9,612,N216FR,DEN,MIA,2255,...,433.0,0.0,0,0,None,None,None,None,None,None
126210,2015,12,31,4,OO,5611,N116SY,LAX,SAN,2300,...,2350.0,0.0,0,0,None,None,None,None,None,None
126211,2015,12,31,4,AS,138,N558AS,ANC,ORD,2300,...,819.0,0.0,0,0,None,None,None,None,None,None


Since 0 does exist for arrival delay, NULL value can't mean there was no arrival delay. Thus, null values for arrival delay must mean something else. Let's run a query to see where arrival delay is null when arrival time exists. 

In [26]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE ARRIVAL_TIME IS NOT NULL AND ARRIVAL_DELAY IS NULL
            ;''', conn)



,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,DL,716,N944DL,OMA,ATL,645,...,1451.0,None,1,0,None,None,None,None,None,None
1,2015,1,1,4,OO,5237,N712SK,MKE,IAH,745,...,1505.0,None,1,0,None,None,None,None,None,None
2,2015,1,1,4,WN,1966,N685SW,ATL,JAX,845,...,1219.0,None,1,0,None,None,None,None,None,None
3,2015,1,1,4,EV,4555,N12552,IAH,HRL,902,...,1319.0,None,1,0,None,None,None,None,None,None
4,2015,1,1,4,WN,1081,N214WN,MDW,OKC,930,...,1343.0,None,1,0,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12553,2015,12,31,4,MQ,3140,N681MQ,DFW,GJT,855,...,1230.0,None,1,0,None,None,None,None,None,None
12554,2015,12,31,4,EV,5059,N855AS,ATL,GNV,1327,...,2328.0,None,1,0,None,None,None,None,None,None
12555,2015,12,31,4,UA,291,N76505,IAD,SMF,1735,...,2216.0,None,1,0,None,None,None,None,None,None
12556,2015,12,31,4,MQ,3060,N648MQ,DFW,FAR,1830,...,2250.0,None,1,0,None,None,None,None,None,None


Look at DIVERTED column. if flight has an arrival time but NULL for arrival delay, there's a 1 in 
DIVERTED column for all the rows (seemingly). For CANCELLED, there is a 0 for all the flights. This must mean if there exists an arrival time but with a NULL arrival delay, the flight was diverted. 

Let's confirm this in the next cell by looking at SUM for DIVERTED column. The WHERE clause in the last cell yielded 12558 rows where there was an arrival time but no arrival delay. If the next query outputs 12558, it confirms this case of flights were all diverted flights. 

In [27]:
pd.read_sql_query('''SELECT SUM(DIVERTED), SUM(CANCELLED)
            FROM flights f
            WHERE ARRIVAL_TIME IS NOT NULL AND ARRIVAL_DELAY IS NULL
            LIMIT 20
            ;''', conn)

,SUM(DIVERTED),SUM(CANCELLED)
0,12558,0


Confirmed. Null values for arrival delay when there exists a value for arrival time means the flight was diverted and was not cancelled entirely. 

**Case 3: Arrival Delay Exists but Arrival Time is Null**

This seems like an odd case to consider but to be thorough in our findings, let's check regardless.

In [28]:
pd.read_sql_query('''SELECT *
            FROM flights f
            WHERE ARRIVAL_TIME IS NULL AND ARRIVAL_DELAY IS NOT NULL
            ;''', conn)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY


So there is no instances where an arrival delay exists but arrival time doesn't. This ensures nothing strange is going on with the data that requires further cleaning/analysis. 

<h5> Arrivals: Aggregations and Joins to Answer Question

Flights that were NOT diverted OR cancelled

Diverted = 1 Cancelled = 1

Not diverted = 0 Not cancelled = 0

Ascending = lowest to highest (highest to lowest for negative values)

In [29]:
#this query is done to successfully carry out the join between flights and airlines tables
pd.read_sql_query('''SELECT 
            a.AIRLINE, f.ARRIVAL_TIME, f.ARRIVAL_DELAY 
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY IS NOT NULL AND ARRIVAL_TIME IS NOT NULL AND DIVERTED == 0 AND CANCELLED == 0
            ORDER BY CAST(ARRIVAL_DELAY AS INTEGER)
           ;''', conn) #limit for 10 rows since flight dataset is massive

,AIRLINE,ARRIVAL_TIME,ARRIVAL_DELAY
0,US Airways Inc.,538.0,-87.0
1,American Airlines Inc.,40.0,-87.0
2,Alaska Airlines Inc.,1927.0,-82.0
3,United Air Lines Inc.,2129.0,-81.0
4,Virgin America,1429.0,-81.0
...,...,...,...
5714003,American Airlines Inc.,1641.0,1636.0
5714004,American Airlines Inc.,1407.0,1638.0
5714005,American Airlines Inc.,1544.0,1665.0
5714006,American Airlines Inc.,1652.0,1898.0


In [30]:
#aggregation query to make steps towards answering question
#averaging only non-zero delays (meaning actual delays, not using flights that arrived early in aggregation)
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.ARRIVAL_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY > 0 AND ARRIVAL_DELAY IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL 
                AND DIVERTED == 0 
                AND CANCELLED == 0
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.ARRIVAL_DELAY) ASC
            ;''', conn)

,AIRLINE,AVG(f.ARRIVAL_DELAY)
0,Hawaiian Airlines Inc.,15.379767
1,Alaska Airlines Inc.,22.562411
2,US Airways Inc.,27.419925
3,Southwest Airlines Co.,29.418496
4,Virgin America,30.725227
5,Delta Air Lines Inc.,32.077424
6,Skywest Airlines Inc.,32.437278
7,American Airlines Inc.,34.148364
8,Atlantic Southeast Airlines,35.198042
9,JetBlue Airways,38.132807


In [31]:
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.ARRIVAL_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY IS NOT NULL AND ARRIVAL_TIME IS NOT NULL AND DIVERTED == 0 AND CANCELLED == 0
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.ARRIVAL_DELAY) ASC
            ;''', conn)

,AIRLINE,AVG(f.ARRIVAL_DELAY)
0,Alaska Airlines Inc.,-0.976563
1,Delta Air Lines Inc.,0.186754
2,Hawaiian Airlines Inc.,2.023093
3,American Airlines Inc.,3.451372
4,US Airways Inc.,3.706209
5,Southwest Airlines Co.,4.374964
6,Virgin America,4.737706
7,United Air Lines Inc.,5.431594
8,Skywest Airlines Inc.,5.845652
9,American Eagle Airlines Inc.,6.457873


In [32]:
pd.read_sql_query('''SELECT 
            a.AIRLINE, MIN(f.ARRIVAL_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY IS NOT NULL AND ARRIVAL_TIME IS NOT NULL AND DIVERTED == 0 AND CANCELLED == 0
            GROUP BY a.AIRLINE
            ORDER BY MIN(f.ARRIVAL_DELAY) ASC
            LIMIT 10;''', conn)

,AIRLINE,MIN(f.ARRIVAL_DELAY)
0,American Airlines Inc.,-87.0
1,US Airways Inc.,-87.0
2,Alaska Airlines Inc.,-82.0
3,United Air Lines Inc.,-81.0
4,Virgin America,-81.0
5,Delta Air Lines Inc.,-79.0
6,JetBlue Airways,-76.0
7,Frontier Airlines Inc.,-73.0
8,Southwest Airlines Co.,-73.0
9,Skywest Airlines Inc.,-69.0


In [33]:
pd.read_sql_query('''SELECT 
            a.AIRLINE, MAX(f.ARRIVAL_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY IS NOT NULL AND ARRIVAL_TIME IS NOT NULL AND DIVERTED == 0 AND CANCELLED == 0
            GROUP BY a.AIRLINE
            ORDER BY MAX(f.ARRIVAL_DELAY) DESC
            LIMIT 10;''', conn)

,AIRLINE,MAX(f.ARRIVAL_DELAY)
0,American Airlines Inc.,1971.0
1,American Eagle Airlines Inc.,1528.0
2,Hawaiian Airlines Inc.,1467.0
3,Skywest Airlines Inc.,1372.0
4,United Air Lines Inc.,1294.0
5,Delta Air Lines Inc.,1274.0
6,Atlantic Southeast Airlines,1223.0
7,Frontier Airlines Inc.,1101.0
8,JetBlue Airways,1002.0
9,Alaska Airlines Inc.,950.0


<h5> Conclusion: </h5>

The following is only for flights that were not cancelled or diverted. 

3 airlines with the least average arrival delay, including arrivals that happened earlier than scheduled, were Hawaiian, Alaskan, and Delta Airlines. 

3 airlines with least average delays for arrivals (counting only pure delays, no early arrivals included) were Hawaiian, Alaskan, and US Airlines. 

3 earliest arrivals that happened before scheduled arrivals was American, US, and Alaska airlines.

3 airlines with the longest arrival delays was American, American Eagle, and Hawaiian airlines. 



<h3> Taxi Time

In [34]:
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.TAXI_OUT) AS AVG_TAXI_TIME
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_TIME IS NOT NULL
                AND CANCELLED == 0
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.TAXI_OUT) ASC;''', conn)

,AIRLINE,AVG_TAXI_TIME
0,Hawaiian Airlines Inc.,10.953483
1,Southwest Airlines Co.,11.943616
2,Spirit Air Lines,14.603710
3,Virgin America,14.755854
4,Alaska Airlines Inc.,15.094197
5,Frontier Airlines Inc.,15.674198
6,American Eagle Airlines Inc.,16.548148
7,Atlantic Southeast Airlines,16.736153
8,United Air Lines Inc.,17.413798
9,Delta Air Lines Inc.,17.608149


In [35]:
pd.read_sql_query('''SELECT 
            a.AIRLINE, COUNT(a.airline)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            GROUP BY a.AIRLINE
            ORDER BY COUNT(a.airline) DESC;''', conn)

,AIRLINE,COUNT(a.airline)
0,Southwest Airlines Co.,1261855
1,Delta Air Lines Inc.,875881
2,American Airlines Inc.,725984
3,Skywest Airlines Inc.,588353
4,Atlantic Southeast Airlines,571977
5,United Air Lines Inc.,515723
6,American Eagle Airlines Inc.,294632
7,JetBlue Airways,267048
8,US Airways Inc.,198715
9,Alaska Airlines Inc.,172521


Final thoughts:

- Hawaiian consistently ranked best across departure and arrival punctuality
- Alaska also performed strongly
- some airlines had low averages but extreme worst-case delays
- diverted/cancelled flights complicate “service quality”
- for a more in-depth analysis, taking a look at columns like TAXI_TIME and comparing AIR_TIME with ELAPSED_time would also answer

<h2> MongoDB Part

In [36]:
import pymongo
from pymongo import MongoClient
import pandas as pd
client = MongoClient('localhost', 27017)
db = client.example

In [37]:
flights_airlines = pd.merge(airlines, flights, left_on='IATA_CODE', right_on = 'AIRLINE', how = 'inner' )
flights_airlines
#a.IATA_CODE = f.AIRLINE

,IATA_CODE,AIRLINE_x,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_y,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,UA,United Air Lines Inc.,2015,1,1,4,UA,1197,N78448,SFO,...,619.0,-7.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,UA,United Air Lines Inc.,2015,1,1,4,UA,1545,N76517,LAX,...,607.0,-11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,UA,United Air Lines Inc.,2015,1,1,4,UA,1528,N76519,SJU,...,458.0,-11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,UA,United Air Lines Inc.,2015,1,1,4,UA,1162,N37293,BQN,...,605.0,6.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,UA,United Air Lines Inc.,2015,1,1,4,UA,1500,N30401,ORD,...,816.0,11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5819074,VX,Virgin America,2015,12,31,4,VX,769,N622VA,LGA,...,2154.0,-6.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819075,VX,Virgin America,2015,12,31,4,VX,357,N284VA,BOS,...,2204.0,-46.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819076,VX,Virgin America,2015,12,31,4,VX,1916,N853VA,SFO,...,2052.0,-18.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819077,VX,Virgin America,2015,12,31,4,VX,490,N840VA,LAX,...,2044.0,-11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
sample = flights_airlines.sample(10000, random_state=1)
sample_dict = sample.to_dict('records')
flights_airlines_collection = db.flights_airlines
flights_airlines_collection.insert_many(sample_dict)

InsertManyResult([ObjectId('6a194b2f53023f793be6136f'), ObjectId('6a194b2f53023f793be61370'), ObjectId('6a194b2f53023f793be61371'), ObjectId('6a194b2f53023f793be61372'), ObjectId('6a194b2f53023f793be61373'), ObjectId('6a194b2f53023f793be61374'), ObjectId('6a194b2f53023f793be61375'), ObjectId('6a194b2f53023f793be61376'), ObjectId('6a194b2f53023f793be61377'), ObjectId('6a194b2f53023f793be61378'), ObjectId('6a194b2f53023f793be61379'), ObjectId('6a194b2f53023f793be6137a'), ObjectId('6a194b2f53023f793be6137b'), ObjectId('6a194b2f53023f793be6137c'), ObjectId('6a194b2f53023f793be6137d'), ObjectId('6a194b2f53023f793be6137e'), ObjectId('6a194b2f53023f793be6137f'), ObjectId('6a194b2f53023f793be61380'), ObjectId('6a194b2f53023f793be61381'), ObjectId('6a194b2f53023f793be61382'), ObjectId('6a194b2f53023f793be61383'), ObjectId('6a194b2f53023f793be61384'), ObjectId('6a194b2f53023f793be61385'), ObjectId('6a194b2f53023f793be61386'), ObjectId('6a194b2f53023f793be61387'), ObjectId('6a194b2f53023f793be613

In [39]:
result = flights_airlines_collection.find_one({'YEAR': 2015})
print(result)

{'_id': ObjectId('6a1931516d08ca1732498ef9'), 'IATA_CODE': 'UA', 'AIRLINE_x': 'United Air Lines Inc.', 'YEAR': 2015, 'MONTH': 1, 'DAY': 1, 'DAY_OF_WEEK': 4, 'AIRLINE_y': 'UA', 'FLIGHT_NUMBER': 1197, 'TAIL_NUMBER': 'N78448', 'ORIGIN_AIRPORT': 'SFO', 'DESTINATION_AIRPORT': 'IAH', 'SCHEDULED_DEPARTURE': 48, 'DEPARTURE_TIME': 42.0, 'DEPARTURE_DELAY': -6.0, 'TAXI_OUT': 11.0, 'WHEELS_OFF': 53.0, 'SCHEDULED_TIME': 218.0, 'ELAPSED_TIME': 217.0, 'AIR_TIME': 199.0, 'DISTANCE': 1635, 'WHEELS_ON': 612.0, 'TAXI_IN': 7.0, 'SCHEDULED_ARRIVAL': 626, 'ARRIVAL_TIME': 619.0, 'ARRIVAL_DELAY': -7.0, 'DIVERTED': 0, 'CANCELLED': 0, 'CANCELLATION_REASON': nan, 'AIR_SYSTEM_DELAY': nan, 'SECURITY_DELAY': nan, 'AIRLINE_DELAY': nan, 'LATE_AIRCRAFT_DELAY': nan, 'WEATHER_DELAY': nan}


<h4> Mongo for Departures Aggregation (Avg)

In [40]:
pipeline = [
    {'$match': {'DEPARTURE_DELAY': {'$gt': 0, '$ne': None},
            'DEPARTURE_TIME': {'$ne': None},
            'ARRIVAL_TIME': {'$ne': None}}},
    {'$group': {'_id': '$AIRLINE_x',
                'Average_Delay': {'$avg': '$DEPARTURE_DELAY'}}},
    {'$sort': {'Average_Delay': 1}}
]
result = flights_airlines_collection.aggregate(pipeline)
for e in result:
    print(e)

{'_id': 'Hawaiian Airlines Inc.', 'Average_Delay': 16.81391330107633}
{'_id': 'Alaska Airlines Inc.', 'Average_Delay': 26.027648401826482}
{'_id': 'Southwest Airlines Co.', 'Average_Delay': 26.949424811225047}
{'_id': 'US Airways Inc.', 'Average_Delay': 28.47483946849768}
{'_id': 'Delta Air Lines Inc.', 'Average_Delay': 29.669210425301888}
{'_id': 'Virgin America', 'Average_Delay': 30.230461092644678}
{'_id': 'United Air Lines Inc.', 'Average_Delay': 32.596844522721454}
{'_id': 'American Airlines Inc.', 'Average_Delay': 34.36750217474864}
{'_id': 'JetBlue Airways', 'Average_Delay': 37.60850839926726}
{'_id': 'Skywest Airlines Inc.', 'Average_Delay': 39.216155698001565}
{'_id': 'American Eagle Airlines Inc.', 'Average_Delay': 40.145680742203254}
{'_id': 'Atlantic Southeast Airlines', 'Average_Delay': 40.83799163963795}
{'_id': 'Spirit Air Lines', 'Average_Delay': 41.900836868049986}
{'_id': 'Frontier Airlines Inc.', 'Average_Delay': 44.50065650510932}


<h4> Mongo for Arrivals Aggregation (Avg)

In [41]:
pipeline = [
    {'$match': {'ARRIVAL_DELAY': {'$gt': 0, '$ne': None},
            'ARRIVAL_TIME': {'$ne': None},
            'DIVERTED': 0,
            'CANCELLED': 0}},
    {'$group': {'_id': '$AIRLINE_x', 
                'Average_Arrival_Delay': {'$avg': '$ARRIVAL_DELAY'}}},
    {'$sort': {'Average_Arrival_Delay': 1}}
]
result = flights_airlines_collection.aggregate(pipeline)
for e in result:
    print(e)

{'_id': 'Hawaiian Airlines Inc.', 'Average_Arrival_Delay': 15.366899597864064}
{'_id': 'Alaska Airlines Inc.', 'Average_Arrival_Delay': 22.553402638245828}
{'_id': 'US Airways Inc.', 'Average_Arrival_Delay': 27.409803793755295}
{'_id': 'Southwest Airlines Co.', 'Average_Arrival_Delay': 29.414379154030666}
{'_id': 'Virgin America', 'Average_Arrival_Delay': 30.66580107992251}
{'_id': 'Delta Air Lines Inc.', 'Average_Arrival_Delay': 32.05905166773077}
{'_id': 'Skywest Airlines Inc.', 'Average_Arrival_Delay': 32.43665279828211}
{'_id': 'American Airlines Inc.', 'Average_Arrival_Delay': 34.13484329630097}
{'_id': 'Atlantic Southeast Airlines', 'Average_Arrival_Delay': 35.19742058864686}
{'_id': 'JetBlue Airways', 'Average_Arrival_Delay': 38.15681988935182}
{'_id': 'United Air Lines Inc.', 'Average_Arrival_Delay': 39.19148924802992}
{'_id': 'American Eagle Airlines Inc.', 'Average_Arrival_Delay': 39.488032988580876}
{'_id': 'Spirit Air Lines', 'Average_Arrival_Delay': 40.664939702098614}
{'_

<h4> Mongo for Taxi Time Aggregation (Avg)

In [42]:
pipeline = [
    {'$match': {'DEPARTURE_TIME': {'$ne': None},
            'TAXI_OUT': {'$type': 'number'}}},
    {'$group': {'_id': '$AIRLINE_x',
            'AVG_TAXI_TIME': {'$avg': '$TAXI_OUT'}}},
    {'$sort': {'AVG_TAXI_TIME': 1}}]
result = flights_airlines_collection.aggregate(pipeline)
for e in result:
    print(e)

{'_id': 'Southwest Airlines Co.', 'AVG_TAXI_TIME': nan}
{'_id': 'Delta Air Lines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Atlantic Southeast Airlines', 'AVG_TAXI_TIME': nan}
{'_id': 'American Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'JetBlue Airways', 'AVG_TAXI_TIME': nan}
{'_id': 'American Eagle Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Frontier Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Alaska Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Hawaiian Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'US Airways Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'United Air Lines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Spirit Air Lines', 'AVG_TAXI_TIME': nan}
{'_id': 'Virgin America', 'AVG_TAXI_TIME': nan}
{'_id': 'Skywest Airlines Inc.', 'AVG_TAXI_TIME': nan}


In [43]:
doc = flights_airlines_collection.find_one(
    {'TAXI_OUT': {'$exists': True}}
)

print(doc['TAXI_OUT'])
print(type(doc['TAXI_OUT']))

11.0
<class 'float'>
